In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps (Unix seconds -> UTC)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Ensure numeric values
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# 1) metric.csv summary: group by (cmdb_id, kpi_name)
metric_summary = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        count=('value', 'count'),
        earliest=('timestamp', 'min'),
        latest=('timestamp', 'max'),
        p95=('value', lambda s: s.quantile(0.95))
    )
    .reset_index()
    .sort_values('count', ascending=False)
    .head(50)  # top 50 by count
)

# 2) trace.csv summary: group by (cmdb_id, trace_name)
trace_summary = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(
        count=('value', 'count'),
        earliest=('timestamp', 'min'),
        latest=('timestamp', 'max'),
        p95=('value', lambda s: s.quantile(0.95))
    )
    .reset_index()
    .sort_values('count', ascending=False)
)
# limit to top 50 if many
trace_summary = trace_summary.head(50)

# 3) log.csv summary: group by (cmdb_id, log_name)
log_summary = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .agg(
        count=('value', 'count'),
        earliest=('timestamp', 'min'),
        latest=('timestamp', 'max'),
        p95=('value', lambda s: s.quantile(0.95))
    )
    .reset_index()
    .sort_values('count', ascending=False)
)
# limit to top 50 if many
log_summary = log_summary.head(50)

# 4) error_logs.csv: total count, earliest, latest, top 10 rows (timestamp, cmdb_id, message)
error_agg = pd.DataFrame([{
    'total_count': len(error_df),
    'earliest': error_df['timestamp'].min() if not error_df.empty else pd.NaT,
    'latest': error_df['timestamp'].max() if not error_df.empty else pd.NaT
}])

error_top10 = error_df[['timestamp', 'cmdb_id', 'message']].sort_values('timestamp').head(10)

# Display compact summaries (variables shown as output)
metric_summary, trace_summary, log_summary, error_agg, error_top10

```
Out[1]:
```
```python
# Compose a concise plain-English summary based on prior summaries in memory.
top_trace = trace_summary.sort_values('p95', ascending=False).iloc[0]
# check if any log.error_count shows non-zero p95
err_logs_nonzero = log_summary[(log_summary['log_name']=='log.error_count') & (log_summary['p95']>0)]
any_errors = not err_logs_nonzero.empty
error_count_total = int(error_agg.loc[0, 'total_count']) if not error_agg.empty else 0

summary = (
    "Summary of findings:\n"
    f"- No raw error log rows found (total_count={error_count_total}).\n"
    f"- Traces: top latency signal is from service '{top_trace['cmdb_id']}' "
    f"({top_trace['trace_name']}) with P95={top_trace['p95']:.6g} s; "
    f"data range {top_trace['earliest']} to {top_trace['latest']}.\n"
    "- Logs: normalized log.error_count is 0 for all services in the summary (no elevated error counts).\n"
    "- Metrics: aggregated metrics do not show obvious widespread error counts; several services show normal CPU/memory/disk metrics at their P95s.\n\n"
    "Conclusion (straightforward):\n"
    f"- The likely root-cause component is '{top_trace['cmdb_id']}' (checkoutservice in the visible data), exhibiting elevated request latency (trace latency P95 high) and high request row_count in traces. "
    "There are no error logs, so this appears to be a performance/latency bottleneck under load rather than an application crash or discrete errors.\n\n"
    "Recommended next steps:\n"
    "- Investigate resource usage (CPU, memory, threads) and backend calls for the '{0}' service during the noted time window; correlate per-instance metrics with traces.\n"
    "- Inspect traces for slow downstream calls and database or external service latency.\n"
    "- If load is high, consider throttling, autoscaling, or optimizing checkoutservice request handling."
).format(top_trace['cmdb_id'])

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id    kpi_name  count                  earliest                    latest           p95
0        adservice         cpu     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00  7.915483e-01
1        adservice      diskio     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00  7.375821e+04
2        adservice  latency-50     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00  3.030805e-03
3        adservice  latency-90     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00  4.670558e-03
4        adservice         mem     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00  1.132010e+08
..             ...         ...    ...                       ...                       ...           ...
45  paymentservice  latency-50     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00  3.550291e-03
46  paymentservice  latency-90     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00  7.585203e-03
47  paymentservice         mem     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00  4.483827e+07
48  paymentservice      socket     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00  3.000000e+00
49  paymentservice    workload     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00  1.073097e+00

[50 rows x 6 columns],             cmdb_id                                trace_name  count                  earliest                    latest         p95
0   checkoutservice  trace.from_checkoutservice.duration_mean     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00    0.005516
1   checkoutservice   trace.from_checkoutservice.duration_p95     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00    0.006688
2   checkoutservice     trace.from_checkoutservice.error_rate     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00    0.000000
3   checkoutservice      trace.from_checkoutservice.row_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00  274.800000
4   checkoutservice  trace.from_frontendservice.duration_mean     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00    0.112053
..              ...                                       ...    ...                       ...                       ...         ...
45     emailservice   trace.from_checkoutservice.duration_p95     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00    0.000570
46     emailservice     trace.from_checkoutservice.error_rate     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00    0.000000
47     emailservice      trace.from_checkoutservice.row_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00   31.800000
48     emailservice             trace.from_root.duration_mean     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00    0.000057
49     emailservice              trace.from_root.duration_p95     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00    0.000109

[50 rows x 6 columns],                   cmdb_id         log_name  count                  earliest                    latest     p95
0               adservice  log.error_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00     0.0
1               adservice    log.row_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00   470.6
2             cartservice  log.error_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00     0.0
3             cartservice    log.row_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00   798.6
4         checkoutservice  log.error_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00     0.0
5         checkoutservice    log.row_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00    95.4
6         currencyservice  log.error_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00     0.0
7         currencyservice    log.row_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00  2465.0
8            emailservice  log.error_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00     0.0
9            emailservice    log.row_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00    31.8
10               frontend  log.error_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00     0.0
11               frontend    log.row_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00  2730.8
12         paymentservice  log.error_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00     0.0
13         paymentservice    log.row_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00    63.6
14  recommendationservice  log.error_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00     0.0
15  recommendationservice    log.row_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00   588.4
19        shippingservice    log.row_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00   480.0
18        shippingservice  log.error_count     25 2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00     0.0
17                  redis    log.row_count      4 2024-01-17 23:03:00+00:00 2024-01-17 23:18:00+00:00     5.0
16                  redis  log.error_count      4 2024-01-17 23:03:00+00:00 2024-01-17 23:18:00+00:00     0.0,    total_count earliest latest
0            0      NaT    NaT, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse metric_df if already in memory, otherwise load
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
else:
    # ensure timestamp parsed
    if not pd.api.types.is_datetime64_any_dtype(metric_df['timestamp']):
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure numeric 'value'
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Candidate services
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]

# Compute global thresholds (p95 and p05) using the entire series for candidate services
thresholds = (
    metric_df[metric_df['cmdb_id'].isin(candidates)]
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(global_p95=lambda s: s.quantile(0.95),
         global_p05=lambda s: s.quantile(0.05))
    .reset_index()
)

# Incident window (UTC)
start = pd.to_datetime("2024-01-17 22:56:13", utc=True)
end = pd.to_datetime("2024-01-17 23:26:13", utc=True)

# Filter metric rows to candidate services within the incident window
window_df = metric_df[
    (metric_df['cmdb_id'].isin(candidates)) &
    (metric_df['timestamp'] >= start) &
    (metric_df['timestamp'] <= end)
].copy()

# Merge global thresholds into windowed data
window_df = window_df.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Flags for anomalies (above p95 or below p05). Handle NaN thresholds safely.
window_df['is_above_p95'] = (window_df['value'] >= window_df['global_p95']) & (~window_df['global_p95'].isna())
window_df['is_below_p05'] = (window_df['value'] <= window_df['global_p05']) & (~window_df['global_p05'].isna())
window_df['is_anom'] = window_df['is_above_p95'] | window_df['is_below_p05']

# Aggregate per (cmdb_id, kpi_name) in the window
agg = window_df.groupby(['cmdb_id', 'kpi_name'], dropna=False).agg(
    total_points_in_window=('value', 'count'),
    count_anomalies_above_p95=('is_above_p95', 'sum'),
    count_anomalies_below_p05=('is_below_p05', 'sum'),
    earliest_anomaly_timestamp=('timestamp', lambda s: s[window_df.loc[s.index, 'is_anom']].min() if s.size>0 else pd.NaT),
    latest_anomaly_timestamp=('timestamp', lambda s: s[window_df.loc[s.index, 'is_anom']].max() if s.size>0 else pd.NaT),
    max_value_in_window=('value', 'max'),
    min_value_in_window=('value', 'min'),
).reset_index()

# Merge back global thresholds for groups that may have zero points in window (ensure all candidate KPI groups represented)
summary = thresholds.merge(agg, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for groups with no points in window
summary['total_points_in_window'] = summary['total_points_in_window'].fillna(0).astype(int)
summary['count_anomalies_above_p95'] = summary['count_anomalies_above_p95'].fillna(0).astype(int)
summary['count_anomalies_below_p05'] = summary['count_anomalies_below_p05'].fillna(0).astype(int)
summary['earliest_anomaly_timestamp'] = pd.to_datetime(summary['earliest_anomaly_timestamp'])
summary['latest_anomaly_timestamp'] = pd.to_datetime(summary['latest_anomaly_timestamp'])
summary['max_value_in_window'] = summary['max_value_in_window'].astype(float)
summary['min_value_in_window'] = summary['min_value_in_window'].astype(float)

# Keep only rows where there is at least one anomaly above p95 or below p05
summary_filtered = summary[
    (summary['count_anomalies_above_p95'] > 0) | (summary['count_anomalies_below_p05'] > 0)
].copy()

# Compute severity ratio for sorting: (max_value_in_window / global_p95), handle divide-by-zero
def compute_ratio(row):
    gp95 = row['global_p95']
    mv = row['max_value_in_window']
    if pd.isna(gp95) or pd.isna(mv):
        return -np.inf
    if gp95 == 0:
        return np.inf if mv > 0 else 0.0
    return mv / gp95

if not summary_filtered.empty:
    summary_filtered['severity_ratio'] = summary_filtered.apply(compute_ratio, axis=1)
else:
    summary_filtered['severity_ratio'] = pd.Series(dtype=float)

# Sort and limit to top 20
summary_filtered = summary_filtered.sort_values(
    by=['count_anomalies_above_p95', 'severity_ratio'],
    ascending=[False, False]
).head(20)

# Select and order requested columns
anomalies_summary = summary_filtered[[
    'cmdb_id',
    'kpi_name',
    'global_p95',
    'global_p05',
    'total_points_in_window',
    'count_anomalies_above_p95',
    'earliest_anomaly_timestamp',
    'latest_anomaly_timestamp',
    'max_value_in_window',
    'min_value_in_window'
]].reset_index(drop=True)

# Distinct list of KPIs available for candidate services
kpi_list = pd.DataFrame({'kpi_name': sorted(thresholds['kpi_name'].unique())})

# Display compact results (variables shown as output)
anomalies_summary, kpi_list

```
Out[2]:
```
Summary (plain English):

- Top anomalous signals:
  - recommendationservice — diskio: all 25 points in the incident window exceed the global P95; observed very large max disk IO (≈171,643). This is the strongest single anomaly by count and magnitude.
  - Multiple services (checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice) — socket: every sample in the window equals or exceeds the global P95 (25/25). This indicates widespread socket-related activity/alerts across services.
  - Several services show isolated high values (2 points each) above global P95 for performance/traffic KPIs:
    - checkoutservice — latency-90, latency-50, workload, cpu
    - recommendationservice — workload, latency-50, latency-90
    - productcatalogservice — mem, cpu
    - currencyservice — latency-90
    - emailservice — workload
  - Some KPIs (diskio, error) have global P95 = 0, making any nonzero measurement show up as an anomaly; these appear for multiple services.

- Distinct KPIs present for the candidate services: cpu, diskio, error, latency-50, latency-90, mem, socket, workload.

- Short interpretation / likely root causes:
  - The most urgent signals point to a combination of I/O and socket issues. recommendationservice shows a pronounced disk IO spike; socket-related metrics are elevated across many services. Checkoutservice also shows elevated latency and workload (and some CPU increase), consistent with a performance bottleneck affecting user-facing flows.
  - This pattern suggests either a shared network/socket problem or an infrastructure-level I/O issue that is impacting multiple services, with recommendationservice and checkoutservice showing the biggest service-level impacts.

- Recommended next steps:
  1. Immediately inspect network/socket health (connection errors, timeouts, file-descriptor/socket limits) across the cluster — especially for the services reporting socket anomalies.
  2. Investigate disk I/O on recommendationservice hosts (high I/O, slow disks, excessive writes/reads); check disk queue, IOPS, and any recent changes.
  3. Correlate the elevated checkoutservice latency/workload with backend calls and traces to identify slow downstream dependencies.
  4. Check instance-level resource usage (CPU, memory) and recent deployments or configuration changes for these services.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name    global_p95    global_p05  total_points_in_window  count_anomalies_above_p95 earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window  min_value_in_window
0   recommendationservice      diskio  0.000000e+00  0.000000e+00                      25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00         1.716433e+05         0.000000e+00
1         checkoutservice      socket  9.000000e+00  9.000000e+00                      25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00         9.000000e+00         9.000000e+00
2         currencyservice      socket  4.000000e+00  4.000000e+00                      25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00         4.000000e+00         4.000000e+00
3            emailservice      socket  3.000000e+00  3.000000e+00                      25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00         3.000000e+00         3.000000e+00
4   productcatalogservice      socket  3.000000e+00  3.000000e+00                      25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00         3.000000e+00         3.000000e+00
5         checkoutservice      diskio  0.000000e+00  0.000000e+00                      25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00         0.000000e+00         0.000000e+00
6            emailservice      diskio  0.000000e+00  0.000000e+00                      25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00         0.000000e+00         0.000000e+00
7   productcatalogservice       error  0.000000e+00  0.000000e+00                      25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00         0.000000e+00         0.000000e+00
8   recommendationservice         mem  3.500177e+08  4.633641e+07                      25                          2  2024-01-17 22:59:00+00:00 2024-01-17 23:21:00+00:00         4.134630e+08         4.629186e+07
9         checkoutservice  latency-90  2.311464e-01  1.963961e-01                      25                          2  2024-01-17 23:02:00+00:00 2024-01-17 23:20:00+00:00         2.696804e-01         1.939638e-01
10        checkoutservice    workload  1.097237e+00  6.722567e-01                      25                          2  2024-01-17 23:02:00+00:00 2024-01-17 23:23:00+00:00         1.266571e+00         6.644167e-01
11           emailservice    workload  1.107231e+00  6.996067e-01                      25                          2  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00         1.242786e+00         6.921833e-01
12        checkoutservice  latency-50  1.348324e-01  6.379835e-02                      25                          2  2024-01-17 23:02:00+00:00 2024-01-17 23:20:00+00:00         1.489672e-01         5.802076e-02
13  recommendationservice    workload  1.959681e+01  1.692523e+01                      25                          2  2024-01-17 23:07:00+00:00 2024-01-17 23:21:00+00:00         2.043990e+01         1.674655e+01
14  recommendationservice  latency-50  7.462795e-02  7.379419e-03                      25                          2  2024-01-17 23:05:00+00:00 2024-01-17 23:23:00+00:00         7.760856e-02         7.348889e-03
15        currencyservice  latency-90  3.698975e-02  1.422166e-02                      25                          2  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00         3.793992e-02         1.222697e-02
16  productcatalogservice         mem  1.061286e+07  1.022745e+07                      25                          2  2024-01-17 23:07:00+00:00 2024-01-17 23:23:00+00:00         1.088375e+07         1.008497e+07
17        checkoutservice         cpu  5.205593e-01  3.440683e-01                      25                          2  2024-01-17 23:07:00+00:00 2024-01-17 23:19:00+00:00         5.333771e-01         3.224669e-01
18  recommendationservice  latency-90  1.698298e+00  9.560515e-03                      25                          2  2024-01-17 23:00:00+00:00 2024-01-17 23:23:00+00:00         1.736791e+00         9.554050e-03
19  productcatalogservice         cpu  3.715848e+00  2.551141e+00                      25                          2  2024-01-17 23:07:00+00:00 2024-01-17 23:23:00+00:00         3.776235e+00         2.358837e+00,      kpi_name
0         cpu
1      diskio
2       error
3  latency-50
4  latency-90
5         mem
6      socket
7    workload)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse trace_df if available, otherwise load
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
else:
    if not pd.api.types.is_datetime64_any_dtype(trace_df['timestamp']):
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Ensure numeric 'value'
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Candidate services
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]

# Compute global P95 and P05 using the full series (for candidate services only)
thresholds = (
    trace_df[trace_df['cmdb_id'].isin(candidates)]
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(global_p95=lambda s: s.quantile(0.95),
         global_p05=lambda s: s.quantile(0.05))
    .reset_index()
)

# Incident window (UTC)
start = pd.to_datetime("2024-01-17 22:56:13", utc=True)
end = pd.to_datetime("2024-01-17 23:26:13", utc=True)

# Filter trace rows to candidate services within the incident window
window_df = trace_df[
    (trace_df['cmdb_id'].isin(candidates)) &
    (trace_df['timestamp'] >= start) &
    (trace_df['timestamp'] <= end)
].copy()

# Merge global thresholds into windowed data
window_df = window_df.merge(thresholds, on=['cmdb_id', 'trace_name'], how='left')

# Flags for anomalies (above p95 or below p05)
window_df['is_above_p95'] = (window_df['value'] >= window_df['global_p95']) & (~window_df['global_p95'].isna())
window_df['is_below_p05'] = (window_df['value'] <= window_df['global_p05']) & (~window_df['global_p05'].isna())
window_df['is_anom'] = window_df['is_above_p95'] | window_df['is_below_p05']

# Aggregate basic stats per group in window
agg_basic = window_df.groupby(['cmdb_id', 'trace_name'], dropna=False).agg(
    total_points_in_window=('value', 'count'),
    count_anomalies_above_p95=('is_above_p95', 'sum'),
    count_anomalies_below_p05=('is_below_p05', 'sum'),
    max_value_in_window=('value', 'max'),
    min_value_in_window=('value', 'min')
).reset_index()

# Earliest/latest anomaly timestamps per group (from anomalous rows only)
anom_rows = window_df[window_df['is_anom']].groupby(['cmdb_id', 'trace_name'], dropna=False).agg(
    earliest_anomaly_timestamp=('timestamp', 'min'),
    latest_anomaly_timestamp=('timestamp', 'max')
).reset_index()

# Merge aggregates with thresholds and anomaly timestamps
summary = thresholds.merge(agg_basic, on=['cmdb_id', 'trace_name'], how='left')
summary = summary.merge(anom_rows, on=['cmdb_id', 'trace_name'], how='left')

# Fill NaNs for groups with no points in window
summary['total_points_in_window'] = summary['total_points_in_window'].fillna(0).astype(int)
summary['count_anomalies_above_p95'] = summary['count_anomalies_above_p95'].fillna(0).astype(int)
summary['count_anomalies_below_p05'] = summary['count_anomalies_below_p05'].fillna(0).astype(int)
summary['max_value_in_window'] = summary['max_value_in_window'].astype(float)
summary['min_value_in_window'] = summary['min_value_in_window'].astype(float)

# Keep only groups that have anomalies (above p95 or below p05)
summary_filtered = summary[
    (summary['count_anomalies_above_p95'] > 0) | (summary['count_anomalies_below_p05'] > 0)
].copy()

# Compute severity ratio for sorting: (max_value_in_window / global_p95)
def compute_ratio(row):
    gp95 = row['global_p95']
    mv = row['max_value_in_window']
    if pd.isna(gp95) or pd.isna(mv):
        return -np.inf
    if gp95 == 0:
        return np.inf if mv > 0 else 0.0
    return mv / gp95

if not summary_filtered.empty:
    summary_filtered['severity_ratio'] = summary_filtered.apply(compute_ratio, axis=1)
else:
    summary_filtered['severity_ratio'] = pd.Series(dtype=float)

# Sort and limit to top 20
summary_filtered = summary_filtered.sort_values(
    by=['count_anomalies_above_p95', 'severity_ratio'],
    ascending=[False, False]
).head(20)

# Select requested columns
anomalies_summary = summary_filtered[[
    'cmdb_id',
    'trace_name',
    'global_p95',
    'global_p05',
    'total_points_in_window',
    'count_anomalies_above_p95',
    'count_anomalies_below_p05',
    'earliest_anomaly_timestamp',
    'latest_anomaly_timestamp',
    'max_value_in_window',
    'min_value_in_window'
]].reset_index(drop=True)

# Distinct list of trace_name values available for candidate services (from full series)
trace_list = pd.DataFrame({'trace_name': sorted(thresholds['trace_name'].unique())})

# Display compact results (variables shown as output)
anomalies_summary, trace_list

```
Out[3]:
```
Summary (plain English):

- What was found
  - The only groups flagged as anomalous in the incident window are many trace error_rate metrics across all candidate services (checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice).
  - For these trace.error_rate groups the computed global P95 and P05 are both 0.0, and the observed values in the window are also 0.0 for every sample (25/25). Because the check used value >= P95 and value <= P05, these zero-valued series are marked as anomalies on both sides.
  - No other trace groups (duration_p95, duration_mean, row_count, etc.) show widespread values exceeding their global P95 or dropping below their global P05 in the provided results.

- Interpretation
  - The flagged “anomalies” are a byproduct of zero-valued error_rate series and zero thresholds, not an obvious sign of increased error traffic. Error rates of 0.0 typically mean no errors were observed; the thresholding logic used here (>= and <= when thresholds are 0) causes these to appear anomalous.
  - There is no strong trace-based signal of elevated latencies or dropped traffic in these candidate services in this window according to the computed P95/P05 results.

- Recommended next steps
  1. Verify semantics and ingestion for trace.error_rate (confirm that 0.0 truly means “no errors” and that traces were recorded correctly).
  2. Adjust anomaly rules to avoid flagging zero-baseline metrics as both high and low (e.g., require strict > P95 for high anomalies, or treat zero-baseline KPIs differently).
  3. If you still suspect performance problems, correlate these trace results with metric findings (e.g., previously observed checkoutservice latency and recommendationservice disk I/O spikes) and inspect duration and row_count traces, service logs, and host-level metrics for those services.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                   trace_name  global_p95  global_p05  total_points_in_window  count_anomalies_above_p95  count_anomalies_below_p05 earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window  min_value_in_window
0         checkoutservice        trace.from_checkoutservice.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
1         checkoutservice        trace.from_frontendservice.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
2         checkoutservice                   trace.from_root.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
3         checkoutservice          trace.to_checkoutservice.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
4         checkoutservice          trace.to_currencyservice.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
5         checkoutservice             trace.to_emailservice.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
6         checkoutservice           trace.to_paymentservice.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
7         checkoutservice    trace.to_productcatalogservice.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
8         currencyservice        trace.from_checkoutservice.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
9         currencyservice        trace.from_frontendservice.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
10        currencyservice                   trace.from_root.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
11           emailservice        trace.from_checkoutservice.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
12           emailservice                   trace.from_root.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
13  productcatalogservice        trace.from_checkoutservice.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
14  productcatalogservice        trace.from_frontendservice.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
15  productcatalogservice  trace.from_recommendationservice.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
16  productcatalogservice                   trace.from_root.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
17  recommendationservice        trace.from_frontendservice.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
18  recommendationservice  trace.from_recommendationservice.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0
19  recommendationservice                   trace.from_root.error_rate         0.0         0.0                      25                         25                         25  2024-01-17 22:59:00+00:00 2024-01-17 23:23:00+00:00                  0.0                  0.0,                                       trace_name
0       trace.from_checkoutservice.duration_mean
1        trace.from_checkoutservice.duration_p95
2          trace.from_checkoutservice.error_rate
3           trace.from_checkoutservice.row_count
4       trace.from_frontendservice.duration_mean
..                                           ...
35      trace.to_productcatalogservice.row_count
36  trace.to_recommendationservice.duration_mean
37   trace.to_recommendationservice.duration_p95
38     trace.to_recommendationservice.error_rate
39      trace.to_recommendationservice.row_count

[40 rows x 1 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse metric_df if available, otherwise load and parse timestamps
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
else:
    if not pd.api.types.is_datetime64_any_dtype(metric_df['timestamp']):
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure numeric values
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Candidate services and KPIs
services = ['recommendationservice','checkoutservice']
kpis = ['diskio','socket','latency-90','latency-50','workload','cpu','mem']

# Incident window (UTC)
start = pd.to_datetime("2024-01-17 22:56:13", utc=True)
end = pd.to_datetime("2024-01-17 23:26:13", utc=True)

# Filter and select requested columns, sort and limit to first 200 rows
selected_samples = (
    metric_df[
        (metric_df['cmdb_id'].isin(services)) &
        (metric_df['kpi_name'].isin(kpis)) &
        (metric_df['timestamp'] >= start) &
        (metric_df['timestamp'] <= end)
    ][['timestamp', 'cmdb_id', 'kpi_name', 'value']]
    .sort_values('timestamp', ascending=True)
    .reset_index(drop=True)
    .head(200)
)

selected_samples

```
Out[4]:
```
Summary (plain English):

- What the filtered samples show (2024-01-17 22:56:13 — 2024-01-17 23:26:13 UTC):
  - recommendationservice:
    - Sustained high workload values (~18–20) across the window.
    - Persistent socket metric at value 4 in every sample.
    - Very large memory samples (examples include 3.54e8 and other high values) — memory spikes are present.
    - diskio is mostly 0 in many samples in this slice, but prior aggregated checks showed large disk IO spikes for this service.
  - checkoutservice:
    - Modest CPU values in the window (example at 22:59 UTC ~0.418).
    - Some workload samples (lower than recommendationservice) and earlier analyses indicated occasional latency/workload spikes for checkoutservice.

- Short interpretation:
  - recommendationservice is the most impacted in this window: sustained high workload plus large memory (and previously observed disk I/O) make it the highest-priority candidate for investigation.
  - checkoutservice shows milder resource/latency signals and is a secondary candidate.

- Recommended next steps:
  1. Inspect recommendationservice hosts for memory pressure and disk I/O (check IOPS, disk latency, swap usage, GC if JVM, and process memory).
  2. Check socket-level metrics and connection counts across the cluster (recommendationservice and any services reporting socket=4).
  3. Correlate these metric samples with traces and logs for the same timestamps to find slow downstream calls or errors.
  4. Consider scaling or restarting affected recommendationservice instances if resource exhaustion is confirmed.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp                cmdb_id    kpi_name         value
0   2024-01-17 22:59:00+00:00        checkoutservice         cpu  4.182865e-01
1   2024-01-17 22:59:00+00:00  recommendationservice    workload  1.879734e+01
2   2024-01-17 22:59:00+00:00  recommendationservice      socket  4.000000e+00
3   2024-01-17 22:59:00+00:00  recommendationservice         mem  4.629186e+07
4   2024-01-17 22:59:00+00:00  recommendationservice  latency-90  9.644665e-03
..                        ...                    ...         ...           ...
195 2024-01-17 23:12:00+00:00        checkoutservice    workload  8.423000e-01
196 2024-01-17 23:13:00+00:00  recommendationservice      diskio  0.000000e+00
197 2024-01-17 23:13:00+00:00  recommendationservice    workload  1.818992e+01
198 2024-01-17 23:13:00+00:00  recommendationservice      socket  4.000000e+00
199 2024-01-17 23:13:00+00:00  recommendationservice         mem  3.544321e+08

[200 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse trace_df if available, otherwise load and parse timestamps
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
else:
    if not pd.api.types.is_datetime64_any_dtype(trace_df['timestamp']):
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Ensure numeric 'value'
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Incident window (UTC)
start = pd.to_datetime("2024-01-17 22:56:13", utc=True)
end = pd.to_datetime("2024-01-17 23:26:13", utc=True)

# Filter rows in window and where trace_name contains 'recommendationservice' (case-sensitive)
recommendation_traces_window = (
    trace_df[
        (trace_df['timestamp'] >= start) &
        (trace_df['timestamp'] <= end) &
        (trace_df['trace_name'].str.contains('recommendationservice', na=False))
    ][['timestamp', 'cmdb_id', 'trace_name', 'value']]
    .sort_values('timestamp', ascending=True)
    .reset_index(drop=True)
    .head(200)
)

recommendation_traces_window

```
Out[5]:
```
summary = (
    "Summary of recommendationservice traces (2024-01-17 22:56:13 — 2024-01-17 23:26:13 UTC):\n"
    "- Many services emit both to_recommendationservice and from_recommendationservice traces in the window.\n"
    "- Trace row_count shows clear traffic spikes to recommendationservice (examples in the samples: 18 requests at 22:59 and a large sample of 571 at 23:08), indicating high request volume.\n"
    "- Trace error_rate values are 0.0 in the sampled rows (no trace-level errors observed).\n"
    "- Latency traces (duration_mean and duration_p95) are small in the sampled rows — no obvious large trace-level latency spikes.\n\n"
    "Interpretation and next steps:\n"
    "- Recommendationservice is receiving substantial traffic but traces do not show increased errors or high latencies; this aligns with metric signals (high workload, memory and disk I/O) seen earlier.\n"
    "- Investigate recommendationservice host-level resources and disk I/O around the high row_count timestamps, and correlate these timestamps with metric spikes to locate resource bottlenecks."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

timestamp                cmdb_id                                      trace_name       value
0   2024-01-17 22:59:00+00:00        frontendservice    trace.to_recommendationservice.duration_mean    0.003724
1   2024-01-17 22:59:00+00:00                   root        trace.to_recommendationservice.row_count   18.000000
2   2024-01-17 22:59:00+00:00                   root       trace.to_recommendationservice.error_rate    0.000000
3   2024-01-17 22:59:00+00:00                   root     trace.to_recommendationservice.duration_p95    0.000080
4   2024-01-17 22:59:00+00:00                   root    trace.to_recommendationservice.duration_mean    0.000054
..                        ...                    ...                                             ...         ...
195 2024-01-17 23:08:00+00:00  recommendationservice   trace.from_recommendationservice.duration_p95    0.004333
196 2024-01-17 23:08:00+00:00  productcatalogservice   trace.from_recommendationservice.duration_p95    0.000028
197 2024-01-17 23:08:00+00:00  productcatalogservice     trace.from_recommendationservice.error_rate    0.000000
198 2024-01-17 23:08:00+00:00  productcatalogservice      trace.from_recommendationservice.row_count  571.000000
199 2024-01-17 23:08:00+00:00  productcatalogservice  trace.from_recommendationservice.duration_mean    0.000017

[200 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```